# 03 — Exploratory Data Analysis
**CareFlow Process Mining | Google Colab**

Explores event counts, case counts, activity frequencies, organization groups, lifecycle transitions, events per case, and event trends over time using the processed sepsis event-log sheet.

In [ ]:
!pip -q install openpyxl pandas numpy matplotlib seaborn pm4py

import io, os, re, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
print("CareFlow Process Mining environment ready.")

In [ ]:
# Upload: Sepsis_Cases_Event_Log_Cleaned_for_Visualization.xlsx
uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload the processed sepsis workbook.")
workbook_name = next(iter(uploaded))
workbook_bytes = uploaded[workbook_name]
xls = pd.ExcelFile(io.BytesIO(workbook_bytes))
print("Workbook:", workbook_name)
print("Sheets:", xls.sheet_names)

required_sheet = "Event Log Visual"
if required_sheet not in xls.sheet_names:
    raise ValueError(f"Expected sheet '{required_sheet}'. Available sheets: {xls.sheet_names}")

events = pd.read_excel(io.BytesIO(workbook_bytes), sheet_name="Event Log Visual")
events.columns = [str(c).strip() for c in events.columns]
print("Event Log Visual shape:", events.shape)
display(events.head())

In [ ]:
# This workbook's event-log sheet uses these verified column names:
CASE_COL = "case_id"
ACTIVITY_COL = "activity"
TIME_COL = "timestamp"
EVENT_INDEX_COL = "event_index"
ORG_COL = "org_group"
LIFECYCLE_COL = "lifecycle_transition"

# Parse timestamp safely; invalid values become NaT and are reported.
if TIME_COL in events.columns:
    events[TIME_COL] = pd.to_datetime(events[TIME_COL], errors="coerce")

print("Case column:", CASE_COL, "| present:", CASE_COL in events.columns)
print("Activity column:", ACTIVITY_COL, "| present:", ACTIVITY_COL in events.columns)
print("Timestamp column:", TIME_COL, "| present:", TIME_COL in events.columns)

## 1. Key metrics

In [ ]:
metrics = pd.DataFrame({
    "metric": ["event_rows", "unique_cases", "unique_activities", "missing_case_ids",
               "missing_activities", "missing_timestamps", "exact_duplicate_rows"],
    "value": [len(events), events[CASE_COL].nunique(dropna=True),
              events[ACTIVITY_COL].nunique(dropna=True), events[CASE_COL].isna().sum(),
              events[ACTIVITY_COL].isna().sum(), events[TIME_COL].isna().sum(),
              events.duplicated().sum()]
})
display(metrics)

## 2. Most frequent activities

In [ ]:
activity_counts = events[ACTIVITY_COL].fillna("Missing activity").value_counts().head(20)
display(activity_counts.rename_axis("activity").reset_index(name="event_count"))
plt.figure(figsize=(11, 7))
activity_counts.sort_values().plot(kind="barh")
plt.title("Top 20 Recorded Activities")
plt.xlabel("Number of events")
plt.ylabel("Activity")
plt.tight_layout()
plt.show()

## 3. Events per case

In [ ]:
events_per_case = events.dropna(subset=[CASE_COL]).groupby(CASE_COL).size().rename("event_count")
display(events_per_case.describe().to_frame())
plt.figure(figsize=(10, 5))
plt.hist(events_per_case, bins=30)
plt.title("Distribution of Events per Case")
plt.xlabel("Events per case")
plt.ylabel("Number of cases")
plt.tight_layout()
plt.show()

## 4. Organization groups and lifecycle transitions

In [ ]:
if ORG_COL in events.columns:
    display(events[ORG_COL].value_counts(dropna=False).head(20).rename_axis("org_group").to_frame("event_count"))
if LIFECYCLE_COL in events.columns:
    display(events[LIFECYCLE_COL].value_counts(dropna=False).rename_axis("lifecycle_transition").to_frame("event_count"))

## 5. Events over time

In [ ]:
time_events = events.dropna(subset=[TIME_COL]).copy()
if not time_events.empty:
    daily = time_events.set_index(TIME_COL).resample("D").size()
    plt.figure(figsize=(12, 5))
    daily.plot()
    plt.title("Recorded Events per Day")
    plt.xlabel("Date")
    plt.ylabel("Event count")
    plt.tight_layout()
    plt.show()
else:
    print("No valid timestamps available for a time trend.")

## 6. Export summary

In [ ]:
metrics.to_csv("eda_key_metrics.csv", index=False)
activity_counts.rename_axis("activity").reset_index(name="event_count").to_csv("activity_frequency.csv", index=False)
events_per_case.rename_axis("case_id").reset_index().to_csv("events_per_case.csv", index=False)
files.download("eda_key_metrics.csv")
files.download("activity_frequency.csv")
files.download("events_per_case.csv")